# Kalman Filter: Interactive Visualization

**Goal:** Implement and visualize the Kalman filter algorithm.

**Time:** 15 minutes

**What you'll build:** A complete Kalman filter that reveals hidden states from noisy observations, with interactive visualizations showing how it works step-by-step.

## Quick Win: Kalman Filter in 40 Lines

Let's implement the complete algorithm and see it work immediately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
np.random.seed(42)

def kalman_filter(y, T, Z, R, Q, H, a0, P0):
    """
    Kalman filter for linear Gaussian state space model.
    
    Returns: filtered states, innovations, innovation variance, log-likelihood
    """
    n = len(y)
    m = len(a0)
    p = y.shape[1] if y.ndim > 1 else 1
    
    # Storage
    a_filt = np.zeros((n, m))
    P_filt = np.zeros((n, m, m))
    v = np.zeros((n, p))
    F = np.zeros((n, p, p))
    loglik = 0.0
    
    # Initialize
    a = a0.copy()
    P = P0.copy()
    
    for t in range(n):
        # Prediction
        a_pred = T @ a
        P_pred = T @ P @ T.T + R @ Q @ R.T
        
        # Handle missing data
        y_t = y[t:t+1].T if y.ndim == 1 else y[t:t+1, :].T
        if not np.isnan(y_t).any():
            # Innovation
            v[t] = (y_t - Z @ a_pred).flatten()
            F[t] = Z @ P_pred @ Z.T + H
            F_inv = np.linalg.inv(F[t])
            
            # Update
            K = P_pred @ Z.T @ F_inv
            a = a_pred + K @ v[t:t+1].T
            a = a.flatten()
            P = P_pred - K @ F[t] @ K.T
            
            # Log-likelihood
            loglik += -0.5 * (p * np.log(2*np.pi) + 
                             np.log(np.linalg.det(F[t])) +
                             v[t] @ F_inv @ v[t])
        else:
            # No update for missing data
            a = a_pred.flatten()
            P = P_pred
        
        a_filt[t] = a
        P_filt[t] = P
    
    return a_filt, P_filt, v, F, loglik

# Test with local level model
T = 100
true_state = np.cumsum(np.random.randn(T) * 0.5)
observations = true_state + np.random.randn(T) * 1.0

# Model matrices
T_mat = np.array([[1.0]])
Z_mat = np.array([[1.0]])
R_mat = np.array([[1.0]])
Q_mat = np.array([[0.25]])
H_mat = np.array([[1.0]])
a0 = np.array([0.0])
P0 = np.array([[10.0]])

# Run filter
a_filt, P_filt, v, F, loglik = kalman_filter(
    observations, T_mat, Z_mat, R_mat, Q_mat, H_mat, a0, P0
)

# Plot
plt.figure(figsize=(14, 5))
plt.plot(true_state, 'k-', linewidth=2, label='True State (hidden)', alpha=0.7)
plt.plot(observations, 'o', color='gray', alpha=0.4, markersize=4, label='Observations')
plt.plot(a_filt, 'r-', linewidth=2, label='Filtered Estimate', alpha=0.8)

# Add confidence bands
std = np.sqrt(P_filt[:, 0, 0])
plt.fill_between(range(T), a_filt.flatten() - 2*std, a_filt.flatten() + 2*std,
                 alpha=0.2, color='red', label='95% Confidence')

plt.legend(loc='best')
plt.title('Kalman Filter: Extracting Signal from Noise')
plt.xlabel('Time')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Log-likelihood: {loglik:.2f}")
print(f"Average estimation error: {np.mean(np.abs(a_filt.flatten() - true_state)):.3f}")
print(f"Average observation error: {np.mean(np.abs(observations - true_state)):.3f}")
print(f"\nFilter reduced error by {(1 - np.mean(np.abs(a_filt.flatten() - true_state)) / np.mean(np.abs(observations - true_state))) * 100:.1f}%")

**See the magic?** The red line (filtered estimate) is much smoother than the noisy observations (gray dots) and closely tracks the true hidden state (black line)!

## How It Works: The Kalman Cycle

The Kalman filter is a two-step dance:

1. **PREDICT**: Use model to forecast where state should be
2. **UPDATE**: When data arrives, blend forecast with observation

The "blend" is determined by the **Kalman gain** K:
- K small → trust model more
- K large → trust data more
- K optimal → best of both worlds!

In [ ]:
def kalman_filter_verbose(y, T, Z, R, Q, H, a0, P0):
    """
    Kalman filter with detailed step-by-step output.
    Returns predictions and Kalman gains too.
    """
    n = len(y)
    m = len(a0)
    
    a_pred = np.zeros((n, m))
    a_filt = np.zeros((n, m))
    K_gains = np.zeros((n, m))
    
    a = a0.copy()
    P = P0.copy()
    
    for t in range(n):
        # PREDICT
        a_pred_t = T @ a
        P_pred = T @ P @ T.T + R @ Q @ R.T
        
        # UPDATE
        v_t = y[t] - Z @ a_pred_t
        F_t = Z @ P_pred @ Z.T + H
        K_t = P_pred @ Z.T / F_t  # Kalman gain (scalar case)
        
        a_filt_t = a_pred_t + K_t * v_t
        P = P_pred - K_t @ F_t @ K_t.T
        
        # Store
        a_pred[t] = a_pred_t.flatten()
        a_filt[t] = a_filt_t.flatten()
        K_gains[t] = K_t.flatten()
        
        a = a_filt_t
    
    return a_pred, a_filt, K_gains

# Run verbose filter
a_pred, a_filt, K_gains = kalman_filter_verbose(
    observations, T_mat, Z_mat, R_mat, Q_mat, H_mat, a0, P0
)

# Visualize the cycle
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Panel 1: Predict vs Update
t_zoom = slice(20, 40)  # Zoom into 20 time periods
axes[0].plot(range(20, 40), observations[t_zoom], 'o', color='gray', 
             markersize=8, label='Observations', alpha=0.6)
axes[0].plot(range(20, 40), a_pred[t_zoom], 's', color='blue', 
             markersize=6, label='Prediction (before update)', alpha=0.7)
axes[0].plot(range(20, 40), a_filt[t_zoom], 'o', color='red', 
             markersize=6, label='Filtered (after update)', alpha=0.8)
axes[0].plot(range(20, 40), true_state[t_zoom], 'k-', 
             linewidth=2, label='True State', alpha=0.5)
axes[0].legend(loc='best')
axes[0].set_title('Predict-Update Cycle (Zoomed In)')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel('Value')

# Panel 2: Kalman Gain Over Time
axes[1].plot(K_gains, linewidth=2, color='purple')
axes[1].set_title('Kalman Gain Over Time (How Much to Trust Data)')
axes[1].set_ylabel('K (Kalman Gain)')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='K=0.5 (equal weight)')
axes[1].legend()

# Panel 3: Innovation (forecast error)
innovations = observations - a_pred.flatten()
axes[2].plot(innovations, linewidth=1, color='green', alpha=0.7)
axes[2].axhline(0, color='red', linestyle='--', linewidth=2)
axes[2].fill_between(range(T), -2*np.std(innovations), 2*np.std(innovations),
                     alpha=0.2, color='green')
axes[2].set_title('Innovations (One-Step-Ahead Forecast Errors)')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Innovation')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial Kalman Gain: {K_gains[0, 0]:.4f}")
print(f"Final Kalman Gain: {K_gains[-1, 0]:.4f}")
print(f"Average Kalman Gain: {K_gains.mean():.4f}")
print(f"\nInterpretation: K≈{K_gains.mean():.2f} means filter weights data and model about equally")

**Key observations:**
1. **Top panel**: Filter estimate (red) is always between prediction (blue) and observation (gray)
2. **Middle panel**: Kalman gain converges quickly to steady-state value
3. **Bottom panel**: Innovations should look like white noise if model is correct

## Interactive: Modify Signal-to-Noise Ratio

Watch how Kalman gain adapts to different noise levels!

In [ ]:
def compare_noise_levels(sigma_state, sigma_obs, T=100):
    """
    Compare Kalman filter performance at different noise levels.
    """
    # Simulate data
    np.random.seed(42)
    true_state = np.cumsum(np.random.randn(T) * sigma_state)
    observations = true_state + np.random.randn(T) * sigma_obs
    
    # Model
    T_mat = np.array([[1.0]])
    Z_mat = np.array([[1.0]])
    R_mat = np.array([[1.0]])
    Q_mat = np.array([[sigma_state**2]])
    H_mat = np.array([[sigma_obs**2]])
    a0 = np.array([0.0])
    P0 = np.array([[10.0]])
    
    # Filter
    a_pred, a_filt, K_gains = kalman_filter_verbose(
        observations, T_mat, Z_mat, R_mat, Q_mat, H_mat, a0, P0
    )
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    axes[0].plot(true_state, 'k-', linewidth=2, label='True State', alpha=0.7)
    axes[0].plot(observations, 'o', color='gray', alpha=0.3, 
                 markersize=3, label='Observations')
    axes[0].plot(a_filt, 'r-', linewidth=2, label='Filtered', alpha=0.8)
    axes[0].legend()
    axes[0].set_title(f'σ_state={sigma_state:.2f}, σ_obs={sigma_obs:.2f}, ' +
                     f'SNR={sigma_state/sigma_obs:.2f}')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(K_gains, linewidth=2, color='purple')
    axes[1].set_title(f'Kalman Gain (steady-state ≈ {K_gains[-20:].mean():.3f})')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('K')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Metrics
    mse_raw = np.mean((observations - true_state)**2)
    mse_filt = np.mean((a_filt.flatten() - true_state)**2)
    improvement = (1 - mse_filt/mse_raw) * 100
    
    print(f"MSE (raw observations): {mse_raw:.3f}")
    print(f"MSE (filtered): {mse_filt:.3f}")
    print(f"Improvement: {improvement:.1f}%")
    print(f"Steady-state Kalman gain: {K_gains[-20:].mean():.4f}")

# Try different scenarios
print("SCENARIO 1: Clean observations (low noise)")
print("="*50)
compare_noise_levels(sigma_state=0.5, sigma_obs=0.2)

print("\n" + "="*50)
print("SCENARIO 2: Moderate noise")
print("="*50)
compare_noise_levels(sigma_state=0.5, sigma_obs=1.0)

print("\n" + "="*50)
print("SCENARIO 3: Very noisy observations")
print("="*50)
compare_noise_levels(sigma_state=0.5, sigma_obs=3.0)

**What did you observe?**

- **Low observation noise** → Large K → Filter trusts data more → Filtered line close to observations
- **High observation noise** → Small K → Filter trusts model more → Filtered line is very smooth
- **Kalman gain automatically adapts** → You don't need to tune anything!

## Real Example: Missing Data

One of the Kalman filter's superpowers: handles missing data naturally!

In [ ]:
# Create data with missing values
np.random.seed(123)
T = 100
true_state = np.cumsum(np.random.randn(T) * 0.5)
observations = true_state + np.random.randn(T) * 1.0

# Randomly remove 30% of observations
missing_mask = np.random.rand(T) < 0.3
observations_missing = observations.copy()
observations_missing[missing_mask] = np.nan

# Run filter
a_filt_missing, P_filt, v, F, loglik = kalman_filter(
    observations_missing, T_mat, Z_mat, R_mat, Q_mat, H_mat, a0, P0
)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full comparison
axes[0].plot(true_state, 'k-', linewidth=2, label='True State', alpha=0.7)
axes[0].plot(observations_missing, 'o', color='gray', alpha=0.5, 
             markersize=4, label='Observations (with gaps)')
axes[0].plot(a_filt_missing, 'r-', linewidth=2, label='Filtered', alpha=0.8)
axes[0].legend()
axes[0].set_title('Kalman Filter with Missing Data (30% missing)')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel('Value')

# Zoom on a gap
gap_region = slice(40, 60)
axes[1].plot(range(40, 60), true_state[gap_region], 'k-', 
             linewidth=3, label='True State')
axes[1].plot(range(40, 60), observations_missing[gap_region], 'o', 
             color='gray', markersize=8, label='Observations')
axes[1].plot(range(40, 60), a_filt_missing[gap_region], 'r-', 
             linewidth=3, label='Filtered')
axes[1].legend()
axes[1].set_title('Zoomed In: Filter Interpolates Through Gaps')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Number of missing observations: {np.sum(missing_mask)}")
print(f"Percentage missing: {np.mean(missing_mask)*100:.1f}%")
print(f"\nFilter still works perfectly!")
print(f"MSE (available data only): {np.mean((a_filt_missing[~missing_mask].flatten() - true_state[~missing_mask])**2):.3f}")

**Magic!** The filter smoothly interpolates through missing data gaps using just the model dynamics. Try doing that with a standard regression!

## Copy-Paste Template: Production Kalman Filter

Ready-to-use code with all the bells and whistles.

In [ ]:
class KalmanFilterEstimator:
    """
    Production-ready Kalman filter with diagnostics.
    """
    
    def __init__(self, T, Z, R, Q, H, a0=None, P0=None):
        self.T = T
        self.Z = Z
        self.R = R
        self.Q = Q
        self.H = H
        
        self.m = T.shape[0]
        self.p = Z.shape[0]
        
        self.a0 = a0 if a0 is not None else np.zeros(self.m)
        self.P0 = P0 if P0 is not None else np.eye(self.m)
        
        self.results = None
    
    def filter(self, y):
        """Run Kalman filter."""
        n = len(y)
        
        # Storage
        a_filt = np.zeros((n, self.m))
        P_filt = np.zeros((n, self.m, self.m))
        a_pred = np.zeros((n, self.m))
        P_pred = np.zeros((n, self.m, self.m))
        v = np.zeros((n, self.p))
        F = np.zeros((n, self.p, self.p))
        K = np.zeros((n, self.m, self.p))
        loglik = 0.0
        
        a = self.a0.copy()
        P = self.P0.copy()
        
        for t in range(n):
            # Prediction
            a_pred[t] = self.T @ a
            P_pred[t] = self.T @ P @ self.T.T + self.R @ self.Q @ self.R.T
            
            # Update
            y_t = y[t:t+1].T if y.ndim == 1 else y[t:t+1, :].T
            
            if not np.isnan(y_t).any():
                v[t] = (y_t - self.Z @ a_pred[t:t+1].T).flatten()
                F[t] = self.Z @ P_pred[t] @ self.Z.T + self.H
                F_inv = np.linalg.inv(F[t])
                K[t] = P_pred[t] @ self.Z.T @ F_inv
                
                a = a_pred[t] + K[t] @ v[t:t+1].T
                a = a.flatten()
                
                # Joseph form (numerically stable)
                I_KZ = np.eye(self.m) - K[t] @ self.Z
                P = I_KZ @ P_pred[t] @ I_KZ.T + K[t] @ self.H @ K[t].T
                
                # Log-likelihood
                loglik += -0.5 * (self.p * np.log(2*np.pi) +
                                 np.log(np.linalg.det(F[t])) +
                                 v[t] @ F_inv @ v[t])
            else:
                a = a_pred[t]
                P = P_pred[t]
            
            a_filt[t] = a
            P_filt[t] = P
        
        self.results = {
            'a_filt': a_filt,
            'P_filt': P_filt,
            'a_pred': a_pred,
            'P_pred': P_pred,
            'v': v,
            'F': F,
            'K': K,
            'loglik': loglik
        }
        
        return self.results
    
    def diagnostics(self):
        """Compute diagnostic statistics."""
        if self.results is None:
            raise ValueError("Run filter() first")
        
        v = self.results['v']
        F = self.results['F']
        
        # Standardized innovations
        v_std = v / np.sqrt(np.diagonal(F, axis1=1, axis2=2))
        
        # Tests
        from scipy.stats import jarque_bera, normaltest
        from statsmodels.stats.diagnostic import acorr_ljungbox
        
        jb_stat, jb_pval = jarque_bera(v_std)
        lb_result = acorr_ljungbox(v_std, lags=10, return_df=False)
        
        diagnostics = {
            'v_std': v_std,
            'v_mean': np.mean(v_std),
            'v_var': np.var(v_std),
            'jb_stat': jb_stat,
            'jb_pval': jb_pval,
            'lb_stat': lb_result[0][-1],
            'lb_pval': lb_result[1][-1]
        }
        
        return diagnostics

# Example usage
kf = KalmanFilterEstimator(T_mat, Z_mat, R_mat, Q_mat, H_mat, a0, P0)
results = kf.filter(observations)
diagnostics = kf.diagnostics()

print("Filter Results:")
print(f"  Log-likelihood: {results['loglik']:.2f}")
print(f"\nDiagnostics:")
print(f"  Innovation mean: {diagnostics['v_mean']:.4f} (should be ≈ 0)")
print(f"  Innovation variance: {diagnostics['v_var']:.4f} (should be ≈ 1)")
print(f"  Jarque-Bera p-value: {diagnostics['jb_pval']:.4f} (> 0.05 is good)")
print(f"  Ljung-Box p-value: {diagnostics['lb_pval']:.4f} (> 0.05 is good)")

if diagnostics['jb_pval'] > 0.05 and diagnostics['lb_pval'] > 0.05:
    print("\n✓ Model passes diagnostic tests!")
else:
    print("\n✗ Model may be misspecified (innovations not white noise)")

## Go Deeper

**Next steps:**
1. Read [Kalman Filter guide](../guides/02_kalman_filter.md) for mathematical details
2. Implement Kalman smoother (retrospective estimation)
3. Apply to real economic data (Module 03)

**Try these extensions:**
- Implement fixed-interval smoother
- Add Kalman gain convergence analysis
- Compare to particle filter (nonlinear case)
- Implement information filter (alternative formulation)

**Key takeaways:**
1. Kalman filter optimally blends model predictions and noisy observations
2. Kalman gain automatically adapts to signal-to-noise ratio
3. Missing data is handled trivially (just skip the update step)
4. Innovations should be white noise if model is correct
5. Algorithm is numerically stable and computationally efficient (O(m³) per step)